# 02 · Model Fitting

Fit the microstructure model, inspect the z-score signal, and sanity-check that signals precede reversion.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd
import plotly.express as px
from src.data_pipeline import DataPipeline
from src.model import MicrostructureModel, build_signal_frame
from src.backtester import EventDrivenBacktester
from src.risk_agent import RiskMonitoringAgent
from src.utils import load_config, CostParams
cfg = load_config(os.path.abspath("../data/config.json"))
cfg

In [ ]:
pipe = DataPipeline(config=cfg); pipe.build()
bars = pipe.load_bars("uniswap_bars"); pipe.close()
train = int(len(bars) * 0.5)
model = MicrostructureModel(cfg["ma_window_bars"], cfg["vol_window_bars"],
                            cfg["z_threshold"], cfg["use_garch"])
model.fit(bars["midprice"].values[:train])
sig = build_signal_frame(bars, model)
sig["signal"].value_counts()

## Z-score with entry thresholds

In [ ]:
import plotly.graph_objects as go
s = sig.dropna().iloc[-1500:]
fig = go.Figure()
fig.add_trace(go.Scatter(x=s["timestamp"], y=s["zscore"], name="z-score"))
fig.add_hline(y=cfg["z_threshold"], line_dash="dash", line_color="red")
fig.add_hline(y=-cfg["z_threshold"], line_dash="dash", line_color="green")
fig.update_layout(title="Rolling z-score vs. entry thresholds", height=350)
fig

## Signal quality: does a BUY precede a price increase?

Forward return over the next N bars, grouped by signal. A working reversion signal shows positive forward returns after BUY and negative after SELL (before costs).

In [ ]:
h = 24  # forward horizon (bars)
sig = sig.copy()
sig["fwd_ret"] = sig["price"].shift(-h) / sig["price"] - 1.0
sig.groupby("signal")["fwd_ret"].agg(["mean", "median", "count"])